In [ ]:
# Cell 1: Setup
import sys
sys.path.append('..')

from src.rag_chain import NetflixGPTRAG, MovieQueryHelper

# Initialize RAG system
print("Initializing Netflix GPT RAG System...")
rag = NetflixGPTRAG(
    model_name="llama3.2",
    temperature=0.7,
    top_k_retrieval=5
)
print("✅ Ready!\n")

In [ ]:
# Cell 2: Basic Recommendation
query = "Recommend an action-packed thriller movie"
response = rag.query(query)

print(f"Question: {response['question']}\n")
print(f"Answer:\n{response['answer']}\n")
print(f"Sources: {[s['title'] for s in response['sources']]}")

In [ ]:
# Cell 3: Mood-Based Query
helper = MovieQueryHelper()
query = helper.format_mood_query("relaxed and want something uplifting")
response = rag.query(query)

print(f"Answer:\n{response['answer']}")

In [ ]:
# Cell 4: Genre + Era Combination
query = helper.format_recommendation_query(
    genre="Science Fiction",
    era="2010s"
)
response = rag.query(query)

print(f"Query: {query}\n")
print(f"Answer:\n{response['answer']}")

In [ ]:
# Cell 5: Similar Movie Query
query = "Movies similar to The Matrix - mind-bending sci-fi"
response = rag.query(query)

print(f"Answer:\n{response['answer']}")

In [ ]:
# Cell 6: Multiple Queries Batch Processing
queries = [
    "Best comedies for date night",
    "Intense psychological thrillers",
    "Feel-good family movies"
]

responses = rag.batch_query(queries)

for resp in responses:
    print(f"\nQ: {resp['question']}")
    print(f"A: {resp['answer'][:200]}...")  # First 200 chars
    print(f"Sources: {len(resp['sources'])} movies")
    print("-" * 50)

In [ ]:
# Cell 7: Analyze Retrieved Context
query = "Award-winning drama films"
retrieval_results = rag.retrieve_context(query)

print(f"Query: {query}\n")
print(f"Retrieved {len(retrieval_results['documents'])} documents:\n")

for i, doc in enumerate(retrieval_results['documents'], 1):
    metadata = doc['metadata']
    score = doc['similarity_score']
    print(f"{i}. {metadata['title']} ({metadata['year']})")
    print(f"   Similarity: {score:.2%}")
    print(f"   Genres: {metadata['genres']}\n")

In [ ]:
# Cell 8: Test Different Temperatures
print("Testing different creativity levels (temperature):\n")

queries_test = ["Recommend a movie for tonight"]

for temp in [0.3, 0.7, 1.0]:
    print(f"\n{'='*60}")
    print(f"Temperature: {temp} ({'focused' if temp < 0.5 else 'balanced' if temp < 0.9 else 'creative'})")
    print('='*60)
    
    rag_temp = NetflixGPTRAG(
        model_name="llama3.2",
        temperature=temp,
        top_k_retrieval=3
    )
    
    response = rag_temp.query(queries_test[0])
    print(f"\n{response['answer'][:300]}...\n")

In [ ]:
# Cell 9: Test Error Handling
print("Testing edge cases:\n")

edge_cases = [
    "",  # Empty query
    "asdfghjkl",  # Nonsense
    "Movies from year 3000",  # Impossible criteria
]

for query in edge_cases:
    if not query:
        print("Skipping empty query")
        continue
    
    try:
        response = rag.query(query)
        print(f"Query: '{query}'")
        print(f"Response: {response['answer'][:150]}...")
        print(f"Sources found: {len(response['sources'])}\n")
    except Exception as e:
        print(f"Query: '{query}' - Error: {e}\n")

In [ ]:
# Cell 10: Performance Timing
import time

test_queries = [
    "Action movies",
    "Romantic comedies", 
    "Sci-fi thrillers",
    "Drama films",
    "Horror movies"
]

print("Performance Test:\n")
times = []

for query in test_queries:
    start = time.time()
    response = rag.query(query, return_sources=False)
    elapsed = time.time() - start
    times.append(elapsed)
    
    print(f"{query:.<30} {elapsed:.2f}s")

print(f"\nAverage query time: {sum(times)/len(times):.2f}s")
print(f"Total time: {sum(times):.2f}s")